# Week 7 — Notebook 3: K-Fold Cross-Validation

**Difficulty:** ⭐⭐⭐ &nbsp;&nbsp; **Estimated Time:** 2.5 hours  
**Domain:** Credit Card Fraud Detection  
**Topics:** k-fold CV algorithm · variance reduction · computational cost · sklearn integration

---

## Why This Matters

After training a fraud-detection model you need to answer one critical question:  
*"How well will this model perform on transactions it has never seen before?"*

A naive approach is to hold out 20% of your data for validation — but with a small dataset that single split is a lottery. You might accidentally put mostly easy cases in the test set (over-optimistic) or mostly hard cases (over-pessimistic). **K-fold cross-validation eliminates the lottery** by rotating the validation window across the entire dataset.

Real-world impact:  
- A bank that over-estimates model performance loses money when the model underperforms in production.  
- A bank that under-estimates rejects a good model and keeps using a worse one.  
- K-fold gives a **stable, unbiased estimate** so neither error occurs.

## Real-World Analogy: The Five-Exam Average

Imagine a professor wants to know how well a student **truly** understands machine learning.  

**Option A — Single exam (holdout validation):**  
One exam, one grade. If the student happens to study the right chapter, they score 95%. If they study the wrong chapter, they score 55%. The single grade is **unreliable** — it depends on which exam was chosen.

**Option B — Five different exams (k-fold CV):**  
Five exams, each covering different material. Average: 82 ± 5%.  
This average is **far more reliable** — no single lucky/unlucky choice dominates.

K-fold cross-validation does exactly this for machine learning models:
- The dataset is divided into **k equal parts (folds)**
- The model is trained and evaluated **k times**, each time using a different fold as the test set
- The **average score ± standard deviation** is reported

```
Fold 1: [TEST ] [train] [train] [train] [train]  → score₁
Fold 2: [train] [TEST ] [train] [train] [train]  → score₂
Fold 3: [train] [train] [TEST ] [train] [train]  → score₃
Fold 4: [train] [train] [train] [TEST ] [train]  → score₄
Fold 5: [train] [train] [train] [train] [TEST ]  → score₅
                                                   ─────────
                          Final score = mean(scores) ± std(scores)
```

**Key property:** Every sample is in the test set **exactly once** → the estimate is unbiased.

## The K-Fold Algorithm

```
INPUT:  model, dataset (X, y), k
OUTPUT: mean score ± std score

1. Shuffle the dataset indices randomly
2. Split into k equal-sized folds
3. FOR fold i in {0, 1, ..., k-1}:
      test_fold  = fold i
      train_fold = all other k-1 folds combined
      model.fit(train_fold)
      score_i = model.evaluate(test_fold)
      scores.append(score_i)
4. RETURN mean(scores), std(scores)
```

### Formula

$$\text{CV Score} = \frac{1}{k} \sum_{i=1}^{k} \text{score}_i \qquad \text{CV Std} = \sqrt{\frac{1}{k-1} \sum_{i=1}^{k}(\text{score}_i - \overline{\text{score}})^2}$$

Always report **both** the mean and the std. A model with mean=0.89, std=0.01 is far more trustworthy than one with mean=0.91, std=0.08.

### Common choices of k
| k | Training data per fold | # Estimates | Computation | Typical use |
|---|---|---|---|---|
| 2 | 50% | 2 | 2× holdout | Almost never used |
| 5 | 80% | 5 | 5× holdout | **Default choice** |
| 10 | 90% | 10 | 10× holdout | When data is plentiful |
| n (LOOCV) | (n-1)/n | n | n× holdout | Small datasets only |

## Setup & Fraud Dataset Creation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    cross_val_score, cross_validate, KFold, LeaveOneOut
)
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
)
from sklearn.datasets import make_classification

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#333',
    'grid.alpha':       0.5,
    'axes.titlecolor':  '#ffffff',
    'figure.titlesize': 14,
})
PALETTE = ['#00d4ff', '#ff6b6b', '#51cf66', '#ffd43b', '#cc5de8', '#ff922b']

print('Libraries loaded successfully.')

In [ ]:
# ── Create the fraud dataset ──────────────────────────────────────────────────
# 3000 credit card transactions, ~3% fraud (90 fraud cases)
# Features: transaction amount, time, merchant category, etc.

np.random.seed(42)
N = 3000
fraud_rate = 0.03
n_fraud = int(N * fraud_rate)          # 90 fraud transactions
n_legit = N - n_fraud                  # 2910 legitimate transactions

# Legitimate transactions
X_legit = np.column_stack([
    np.random.normal(50,  30,  n_legit),   # avg_amount ($)
    np.random.normal(12,  6,   n_legit),   # hour_of_day
    np.random.normal(3,   1.5, n_legit),   # transactions_per_day
    np.random.normal(0.2, 0.1, n_legit),   # foreign_merchant_ratio
    np.random.normal(1,   0.5, n_legit),   # velocity_score
])

# Fraudulent transactions (different distribution)
X_fraud = np.column_stack([
    np.random.normal(180, 80,  n_fraud),   # higher amounts
    np.random.normal(3,   2,   n_fraud),   # unusual hours (late night)
    np.random.normal(8,   3,   n_fraud),   # many transactions in short time
    np.random.normal(0.7, 0.2, n_fraud),   # mostly foreign merchants
    np.random.normal(4,   1,   n_fraud),   # high velocity
])

X = np.vstack([X_legit, X_fraud])
y = np.array([0]*n_legit + [1]*n_fraud)

# Shuffle and scale
shuffle_idx = np.random.permutation(N)
X, y = X[shuffle_idx], y[shuffle_idx]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

feature_names = ['avg_amount', 'hour_of_day', 'txn_per_day',
                 'foreign_ratio', 'velocity_score']

df = pd.DataFrame(X, columns=feature_names)
df['is_fraud'] = y

print(f'Dataset shape: {X.shape}')
print(f'Fraud transactions: {y.sum()} ({y.mean()*100:.1f}%)')
print(f'Legit transactions: {(y==0).sum()} ({(y==0).mean()*100:.1f}%)')
print()
print(df.describe().round(2))

## Part 1 — K-Fold Cross-Validation From Scratch

In [ ]:
# ── K-Fold CV implemented from scratch ───────────────────────────────────────
# We build this ourselves before using sklearn, so you understand every step.

def compute_metric(model, X_test, y_test, metric='accuracy'):
    """Compute a single evaluation metric."""
    y_pred = model.predict(X_test)
    if metric == 'accuracy':
        return accuracy_score(y_test, y_pred)
    elif metric == 'f1':
        return f1_score(y_test, y_pred, zero_division=0)
    elif metric == 'roc_auc':
        y_prob = model.predict_proba(X_test)[:, 1]
        return roc_auc_score(y_test, y_prob)
    else:
        raise ValueError(f'Unknown metric: {metric}')


def kfold_cv(model, X, y, k=5, metric='accuracy', random_state=42):
    """
    K-fold cross-validation from scratch.

    Parameters
    ----------
    model        : sklearn-compatible estimator
    X, y         : feature matrix and labels (numpy arrays)
    k            : number of folds
    metric       : 'accuracy', 'f1', or 'roc_auc'
    random_state : for reproducibility

    Returns
    -------
    np.array of k scores
    """
    rng = np.random.RandomState(random_state)
    n = len(X)
    fold_size = n // k                          # samples per fold
    indices = np.arange(n)
    rng.shuffle(indices)                        # randomise order

    scores = []
    for i in range(k):
        # Identify test indices for fold i
        test_start  = i * fold_size
        test_end    = (i + 1) * fold_size if i < k - 1 else n  # last fold takes remainder
        test_idx    = indices[test_start:test_end]
        train_idx   = np.concatenate([indices[:test_start], indices[test_end:]])

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)             # train on k-1 folds
        score = compute_metric(model, X_test, y_test, metric)
        scores.append(score)
        print(f'  Fold {i+1}/{k}: {metric}={score:.4f}  '
              f'(train={len(train_idx)}, test={len(test_idx)})')

    return np.array(scores)


# Run it
model_lr = LogisticRegression(max_iter=1000, random_state=42)
print('=== K-Fold CV (k=5) — Logistic Regression ===')
scores_scratch = kfold_cv(model_lr, X_scaled, y, k=5, metric='roc_auc')
print(f'\nROC-AUC: {scores_scratch.mean():.4f} ± {scores_scratch.std():.4f}')

## Part 2 — Verify Against sklearn

In [ ]:
# ── Compare scratch implementation vs sklearn cross_val_score ─────────────────
# They should produce identical scores (same random_state, same splits).

# sklearn KFold with same parameters
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model_lr2 = LogisticRegression(max_iter=1000, random_state=42)

scores_sklearn = cross_val_score(
    model_lr2, X_scaled, y, cv=kf, scoring='roc_auc'
)

print('=== Comparison: Scratch vs sklearn ===')
print(f'{"Fold":<6} {"Scratch":>10} {"sklearn":>10} {"Match?":>8}')
print('-' * 38)
for i, (s, sk) in enumerate(zip(scores_scratch, scores_sklearn)):
    match = '✓' if abs(s - sk) < 1e-6 else '✗'
    print(f'{i+1:<6} {s:>10.6f} {sk:>10.6f} {match:>8}')

print()
print(f'Scratch mean  : {scores_scratch.mean():.6f}')
print(f'sklearn mean  : {scores_sklearn.mean():.6f}')

# cross_validate returns multiple metrics at once
print('\n=== cross_validate — multiple metrics simultaneously ===')
cv_results = cross_validate(
    LogisticRegression(max_iter=1000, random_state=42),
    X_scaled, y, cv=kf,
    scoring=['accuracy', 'f1', 'roc_auc']
)
for metric_key in ['test_accuracy', 'test_f1', 'test_roc_auc']:
    vals = cv_results[metric_key]
    print(f'  {metric_key:<20}: {vals.mean():.4f} ± {vals.std():.4f}')

## Part 3 — Variance Reduction: Single Split vs 5-Fold CV

**The core argument for k-fold CV** is that it has lower variance than a single holdout split.  
Let's prove this empirically by running 100 experiments.

In [ ]:
# ── Variance experiment: 100 random 80/20 splits vs 100 runs of 5-fold CV ────
# For each of 100 seeds:
#   (a) Do a single 80/20 random split → record one accuracy
#   (b) Do 5-fold CV → record the mean accuracy
# Compare: which method gives more stable (lower std) estimates?

from sklearn.model_selection import train_test_split

n_experiments = 100
single_split_scores = []
kfold_mean_scores   = []

for seed in range(n_experiments):
    # (a) Single 80/20 holdout split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_scaled, y, test_size=0.2, random_state=seed
    )
    m = LogisticRegression(max_iter=1000, random_state=42)
    m.fit(X_tr, y_tr)
    single_split_scores.append(roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]))

    # (b) 5-fold CV (mean of 5 scores)
    kf_seed = KFold(n_splits=5, shuffle=True, random_state=seed)
    fold_scores = cross_val_score(
        LogisticRegression(max_iter=1000, random_state=42),
        X_scaled, y, cv=kf_seed, scoring='roc_auc'
    )
    kfold_mean_scores.append(fold_scores.mean())

single_split_scores = np.array(single_split_scores)
kfold_mean_scores   = np.array(kfold_mean_scores)

print(f'Single 80/20 split — mean={single_split_scores.mean():.4f},  '
      f'std={single_split_scores.std():.4f}')
print(f'5-fold CV mean     — mean={kfold_mean_scores.mean():.4f},  '
      f'std={kfold_mean_scores.std():.4f}')
print()
reduction = (1 - kfold_mean_scores.std() / single_split_scores.std()) * 100
print(f'Variance reduction from using 5-fold CV: {reduction:.1f}%')

In [ ]:
# ── Plot: Distribution of 100 AUC estimates ───────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Variance Reduction: Single Split vs 5-Fold CV\n'
             '(100 experiments, Logistic Regression, ROC-AUC)',
             fontsize=13, fontweight='bold')

# Left: overlapping histograms
ax = axes[0]
ax.hist(single_split_scores, bins=20, alpha=0.7,
        color=PALETTE[1], label=f'Single split  std={single_split_scores.std():.4f}',
        edgecolor='white', linewidth=0.3)
ax.hist(kfold_mean_scores,   bins=20, alpha=0.7,
        color=PALETTE[0], label=f'5-fold CV mean std={kfold_mean_scores.std():.4f}',
        edgecolor='white', linewidth=0.3)
ax.axvline(single_split_scores.mean(), color=PALETTE[1], lw=2, ls='--')
ax.axvline(kfold_mean_scores.mean(),   color=PALETTE[0], lw=2, ls='--')
ax.set_xlabel('ROC-AUC Score')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of 100 AUC Estimates')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: experiment-by-experiment scatter
ax2 = axes[1]
ax2.plot(range(n_experiments), single_split_scores,
         color=PALETTE[1], alpha=0.6, lw=1, label='Single split')
ax2.plot(range(n_experiments), kfold_mean_scores,
         color=PALETTE[0], alpha=0.9, lw=1.5, label='5-fold CV mean')
ax2.set_xlabel('Experiment index')
ax2.set_ylabel('ROC-AUC')
ax2.set_title('Score per Experiment (ordered by seed)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kfold_variance_reduction.png', dpi=120, bbox_inches='tight')
plt.show()
print('Key insight: the orange line (5-fold CV) is much smoother than the blue line.')

## Part 4 — Effect of k on Bias and Variance

In [ ]:
# ── Sweep k from 2 to 20; record mean and std of CV scores ───────────────────
# Larger k → more training data per fold (lower bias)
# Larger k → smaller test folds & more correlated folds (higher variance in estimate)

k_values = [2, 3, 5, 7, 10, 15, 20]
mean_scores = []
std_scores  = []

for k in k_values:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = cross_val_score(
        LogisticRegression(max_iter=1000, random_state=42),
        X_scaled, y, cv=kf, scoring='roc_auc'
    )
    mean_scores.append(scores.mean())
    std_scores.append(scores.std())
    pct_train = (k - 1) / k * 100
    print(f'k={k:>2}: AUC={scores.mean():.4f} ± {scores.std():.4f}  '
          f'(each model sees {pct_train:.0f}% of data)')

print()
print('Interpretation:')
print('  Mean AUC rises as k increases (more training data = less bias)')
print('  Std may increase at very high k (tiny test folds, noisy individual scores)')

In [ ]:
# ── Plot: mean ± std vs k ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Effect of k on CV Score Bias and Variance', fontsize=13, fontweight='bold')

# Left: mean AUC
ax = axes[0]
ax.plot(k_values, mean_scores, 'o-', color=PALETTE[0], lw=2, ms=8)
ax.fill_between(
    k_values,
    np.array(mean_scores) - np.array(std_scores),
    np.array(mean_scores) + np.array(std_scores),
    alpha=0.25, color=PALETTE[0], label='±1 std'
)
ax.set_xlabel('k (number of folds)')
ax.set_ylabel('Mean ROC-AUC')
ax.set_title('Mean CV Score vs k')
ax.set_xticks(k_values)
ax.legend()
ax.grid(True, alpha=0.3)

# Right: std of fold scores
ax2 = axes[1]
ax2.bar([str(k) for k in k_values], std_scores,
        color=PALETTE[1], alpha=0.85, edgecolor='white')
ax2.set_xlabel('k (number of folds)')
ax2.set_ylabel('Std of Fold Scores')
ax2.set_title('Score Variability Across Folds vs k')
ax2.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(std_scores):
    ax2.text(i, v + 0.0005, f'{v:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('kfold_k_selection.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 5 — Computational Cost Analysis

In [ ]:
# ── Time single fit vs 5-fold vs 10-fold as dataset size grows ───────────────
# Shows that k-fold is exactly k× more expensive (linear scaling).

dataset_sizes = [500, 1000, 2000, 3000]
time_single   = []
time_5fold    = []
time_10fold   = []

for n_samples in dataset_sizes:
    # Sub-sample the dataset
    idx = np.random.choice(N, n_samples, replace=False)
    X_sub, y_sub = X_scaled[idx], y[idx]

    # Single fit (train on 80%)
    split = int(0.8 * n_samples)
    t0 = time.perf_counter()
    m = LogisticRegression(max_iter=1000, random_state=42)
    m.fit(X_sub[:split], y_sub[:split])
    time_single.append(time.perf_counter() - t0)

    # 5-fold CV
    t0 = time.perf_counter()
    cross_val_score(
        LogisticRegression(max_iter=1000, random_state=42),
        X_sub, y_sub,
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc'
    )
    time_5fold.append(time.perf_counter() - t0)

    # 10-fold CV
    t0 = time.perf_counter()
    cross_val_score(
        LogisticRegression(max_iter=1000, random_state=42),
        X_sub, y_sub,
        cv=KFold(n_splits=10, shuffle=True, random_state=42),
        scoring='roc_auc'
    )
    time_10fold.append(time.perf_counter() - t0)

    print(f'n={n_samples:>5}: single={time_single[-1]*1000:.1f}ms  '
          f'5-fold={time_5fold[-1]*1000:.1f}ms  '
          f'10-fold={time_10fold[-1]*1000:.1f}ms  '
          f'(5-fold ratio={time_5fold[-1]/time_single[-1]:.1f}x)')

In [ ]:
# ── Plot computational cost ───────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Computational Cost: Single Fit vs K-Fold CV', fontsize=13, fontweight='bold')

ax.plot(dataset_sizes, [t*1000 for t in time_single],
        'o-', color=PALETTE[2], lw=2, ms=8, label='Single fit (80/20)')
ax.plot(dataset_sizes, [t*1000 for t in time_5fold],
        's-', color=PALETTE[0], lw=2, ms=8, label='5-fold CV (~5x)')
ax.plot(dataset_sizes, [t*1000 for t in time_10fold],
        '^-', color=PALETTE[1], lw=2, ms=8, label='10-fold CV (~10x)')

ax.set_xlabel('Dataset Size (n samples)')
ax.set_ylabel('Wall-clock Time (ms)')
ax.set_title('Time vs Dataset Size')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Annotate ratio at largest dataset
ratio5  = time_5fold[-1]  / time_single[-1]
ratio10 = time_10fold[-1] / time_single[-1]
ax.annotate(f'{ratio5:.1f}x', xy=(dataset_sizes[-1], time_5fold[-1]*1000),
            xytext=(dataset_sizes[-1]-400, time_5fold[-1]*1000*1.15),
            color=PALETTE[0], fontsize=10)
ax.annotate(f'{ratio10:.1f}x', xy=(dataset_sizes[-1], time_10fold[-1]*1000),
            xytext=(dataset_sizes[-1]-400, time_10fold[-1]*1000*1.1),
            color=PALETTE[1], fontsize=10)

plt.tight_layout()
plt.savefig('kfold_cost.png', dpi=120, bbox_inches='tight')
plt.show()

print('Nested CV note: outer 5-fold × inner 5-fold = 25 model fits.')
print('Grid search (m=100 params) × 5-fold = 500 model fits.')

## Part 6 — Repeated K-Fold Cross-Validation

**Idea:** Run k-fold CV multiple times, each with a different random shuffle.  
Produces r×k scores → even tighter confidence interval on the mean.  
Trade-off: r times more computation.

In [ ]:
# ── Repeated 5×5 CV vs standard 5-fold ───────────────────────────────────────
# 5 repetitions × 5 folds = 25 model fits → 25 scores
# Compare confidence intervals: repeated CV should be much tighter.

from sklearn.model_selection import RepeatedKFold

# Standard 5-fold (run once) — 5 scores
kf_standard = KFold(n_splits=5, shuffle=True, random_state=42)
scores_standard = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_scaled, y, cv=kf_standard, scoring='roc_auc'
)

# Repeated 5-fold, 5 repetitions — 25 scores
rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
scores_repeated = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_scaled, y, cv=rkf, scoring='roc_auc'
)

# 95% confidence interval = mean ± 1.96 * std / sqrt(n)
ci_std = 1.96 * scores_standard.std() / np.sqrt(len(scores_standard))
ci_rep = 1.96 * scores_repeated.std() / np.sqrt(len(scores_repeated))

print('=== Standard 5-fold CV (5 scores) ===')
print(f'Scores : {scores_standard.round(4)}')
print(f'Mean   : {scores_standard.mean():.4f}')
print(f'Std    : {scores_standard.std():.4f}')
print(f'95% CI : [{scores_standard.mean()-ci_std:.4f}, {scores_standard.mean()+ci_std:.4f}]')
print(f'CI width: {2*ci_std:.4f}')
print()
print('=== Repeated 5×5 CV (25 scores) ===')
print(f'Scores : {scores_repeated.round(4)}')
print(f'Mean   : {scores_repeated.mean():.4f}')
print(f'Std    : {scores_repeated.std():.4f}')
print(f'95% CI : [{scores_repeated.mean()-ci_rep:.4f}, {scores_repeated.mean()+ci_rep:.4f}]')
print(f'CI width: {2*ci_rep:.4f}')
print()
print(f'CI narrowed by {(1-2*ci_rep/(2*ci_std))*100:.1f}% thanks to repeated CV.')

## Part 7 — Box Plot Comparison: Single Holdout vs 5-Fold vs 10-Fold vs LOOCV

In [ ]:
# ── Compare score distributions using a 200-sample subset ────────────────────
# LOOCV is only practical on small datasets (200 fits).
# We run 50 repetitions of each method to get enough points for a box plot.

np.random.seed(0)
N_SMALL = 200                          # small subset for LOOCV feasibility
N_REPS  = 50                           # repetitions per method

holdout_all = []
kf5_all     = []
kf10_all    = []
loocv_all   = []

for rep in range(N_REPS):
    rng = np.random.RandomState(rep)
    idx = rng.choice(N, N_SMALL, replace=False)
    Xs, ys = X_scaled[idx], y[idx]

    model_rep = LogisticRegression(max_iter=1000, random_state=42)

    # Single holdout (70/30 split)
    split = int(0.7 * N_SMALL)
    p = rng.permutation(N_SMALL)
    Xs_p, ys_p = Xs[p], ys[p]
    model_rep.fit(Xs_p[:split], ys_p[:split])
    try:
        holdout_all.append(roc_auc_score(ys_p[split:],
                            model_rep.predict_proba(Xs_p[split:])[:, 1]))
    except Exception:
        holdout_all.append(np.nan)

    # 5-fold CV
    sc5 = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                          Xs, ys, cv=KFold(5, shuffle=True, random_state=rep),
                          scoring='roc_auc')
    kf5_all.append(sc5.mean())

    # 10-fold CV
    sc10 = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                           Xs, ys, cv=KFold(10, shuffle=True, random_state=rep),
                           scoring='roc_auc')
    kf10_all.append(sc10.mean())

    # LOOCV
    sc_loo = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                             Xs, ys, cv=LeaveOneOut(), scoring='roc_auc')
    loocv_all.append(sc_loo.mean())

print(f'Method       Mean AUC   Std AUC')
print(f'Holdout      {np.nanmean(holdout_all):.4f}     {np.nanstd(holdout_all):.4f}')
print(f'5-fold CV    {np.mean(kf5_all):.4f}     {np.std(kf5_all):.4f}')
print(f'10-fold CV   {np.mean(kf10_all):.4f}     {np.std(kf10_all):.4f}')
print(f'LOOCV        {np.mean(loocv_all):.4f}     {np.std(loocv_all):.4f}')

In [ ]:
# ── Box plot ──────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(11, 6))
fig.suptitle('Stability of AUC Estimates: 50 Repetitions on 200-Sample Subset',
             fontsize=13, fontweight='bold')

data_bp  = [holdout_all, kf5_all, kf10_all, loocv_all]
labels_bp = ['Single\nHoldout', '5-Fold\nCV', '10-Fold\nCV', 'LOOCV']
colors_bp = [PALETTE[1], PALETTE[0], PALETTE[2], PALETTE[3]]

bp = ax.boxplot(data_bp, patch_artist=True, notch=False,
                medianprops={'color': 'white', 'lw': 2},
                whiskerprops={'color': '#aaa'},
                capprops={'color': '#aaa'},
                flierprops={'marker': 'o', 'markerfacecolor': '#aaa',
                            'markersize': 4, 'alpha': 0.5})

for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

# Annotate std
for i, d in enumerate(data_bp):
    s = np.nanstd(d)
    ax.text(i+1, np.nanmax(d)+0.005, f'std={s:.4f}',
            ha='center', va='bottom', fontsize=9, color=colors_bp[i])

ax.set_xticklabels(labels_bp)
ax.set_ylabel('ROC-AUC')
ax.set_title('Lower box height = more stable estimates')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('kfold_boxplot_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 8 — Summary Dashboard

In [ ]:
# ── Summary: everything we learned in one dashboard ───────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('K-Fold Cross-Validation — Summary Dashboard\nCredit Card Fraud Detection',
             fontsize=14, fontweight='bold')

# ① CV algorithm diagram (text in plot)
ax0 = axes[0, 0]
ax0.set_xlim(0, 10)
ax0.set_ylim(0, 7)
ax0.set_title('① K-Fold Algorithm (k=5)', fontsize=11)
ax0.axis('off')

row_labels = [f'Fold {i+1}' for i in range(5)]
colors_map = np.array([[0]*5 for _ in range(5)], dtype=float)
for i in range(5):
    colors_map[i, i] = 1.0                  # 1 = test fold

for row in range(5):
    for col in range(5):
        facecolor = PALETTE[1] if colors_map[row, col] == 1 else PALETTE[0]
        rect = mpatches.FancyBboxPatch(
            (1 + col*1.6, 5.5 - row*1.0), 1.4, 0.7,
            boxstyle='round,pad=0.05',
            facecolor=facecolor, edgecolor='white', alpha=0.8
        )
        ax0.add_patch(rect)
        label = 'TEST' if colors_map[row, col] == 1 else 'train'
        fs = 8 if label == 'train' else 9
        fw = 'normal' if label == 'train' else 'bold'
        ax0.text(1.7 + col*1.6, 5.85 - row*1.0, label,
                 ha='center', va='center', fontsize=fs, fontweight=fw, color='white')
    ax0.text(0.5, 5.85 - row*1.0, row_labels[row],
             ha='center', va='center', fontsize=9, color='#e0e0e0')
ax0.text(5, 0.4, 'Final: mean ± std of 5 scores', ha='center',
         fontsize=10, color='#ffd43b', fontweight='bold')

# ② Variance reduction
ax1 = axes[0, 1]
ax1.hist(single_split_scores, bins=18, alpha=0.7, color=PALETTE[1],
         label=f'Single split (std={single_split_scores.std():.3f})', edgecolor='white', lw=0.3)
ax1.hist(kfold_mean_scores,   bins=18, alpha=0.7, color=PALETTE[0],
         label=f'5-fold CV    (std={kfold_mean_scores.std():.3f})',   edgecolor='white', lw=0.3)
ax1.set_xlabel('ROC-AUC')
ax1.set_ylabel('Frequency')
ax1.set_title('② Variance Reduction (100 experiments)', fontsize=11)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# ③ k selection
ax2 = axes[1, 0]
ax2.errorbar(k_values, mean_scores, yerr=std_scores,
             fmt='o-', color=PALETTE[0], capsize=5, capthick=1.5,
             lw=2, ms=7, ecolor=PALETTE[1])
ax2.axvline(5,  color=PALETTE[2], ls='--', lw=1.5, label='k=5  (recommended)')
ax2.axvline(10, color=PALETTE[3], ls='--', lw=1.5, label='k=10 (common)')
ax2.set_xlabel('k')
ax2.set_ylabel('Mean ROC-AUC ± std')
ax2.set_title('③ Effect of k on Score (mean ± std)', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(k_values)

# ④ Cost comparison
ax3 = axes[1, 1]
methods = ['Single\nFit', '5-Fold\nCV', '10-Fold\nCV']
times_ms = [
    time_single[-1]*1000,
    time_5fold[-1]*1000,
    time_10fold[-1]*1000,
]
bars = ax3.bar(methods, times_ms, color=[PALETTE[2], PALETTE[0], PALETTE[1]],
               alpha=0.85, edgecolor='white')
ax3.set_ylabel('Time (ms)')
ax3.set_title('④ Computational Cost (n=3000)', fontsize=11)
for bar, val in zip(bars, times_ms):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.2,
             f'{val:.1f}ms', ha='center', va='bottom', fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('kfold_summary_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

## Key Takeaways

| Concept | Plain English |
|---|---|
| **K-fold CV** | Rotate test window k times; each sample is tested exactly once |
| **Report mean ± std** | Mean tells you expected performance; std tells you how reliable that estimate is |
| **k=5 is default** | 80% training data, 5 estimates — the best all-round balance |
| **Variance reduction** | 5-fold CV mean has much lower std than a single holdout split |
| **Cost** | k-fold CV costs exactly k× a single fit — plan accordingly |
| **Repeated CV** | r×k fits, r×k scores → CI narrows by ~√r |
| **When to use LOOCV** | Only when n < ~100; otherwise too expensive and high variance |

### Practical Checklist for Fraud Detection
1. Always **shuffle** before splitting (transactions are often time-ordered)
2. Use **`cross_val_score(..., scoring='roc_auc')`** not accuracy for imbalanced data
3. Report **mean ± std**, not just mean
4. For hyperparameter tuning: use **nested CV** (outer for eval, inner for tuning)
5. After selecting the final model, **re-train on all data** before deployment

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1**  
Your 5-fold CV reports mean ROC-AUC = 0.89, std = 0.04. What does the std = 0.04 tell you?

<details>
<summary>Click to reveal answer</summary>

The std = 0.04 means the individual fold scores varied by about ±0.04 around the mean. In practice, the 5 scores ranged roughly from 0.85 to 0.93. This tells you **how sensitive the model is to which data it sees during training**. A low std (e.g., 0.01) means the model is robust and consistently performs well regardless of the training split. A high std (e.g., 0.08) means the model is unstable — maybe the dataset is small, or the model is overfitting some folds. You should **be cautious** about deploying a model with high std even if the mean looks good.

</details>

---

**Question 2**  
You have a fraud dataset of 100 samples. Should you use k=5 or k=100 (LOOCV)? What are the tradeoffs?

<details>
<summary>Click to reveal answer</summary>

With n=100, both are viable, but each has tradeoffs:

| | k=5 | k=100 (LOOCV) |
|---|---|---|
| Training size | 80 samples | 99 samples |
| Test size | 20 samples | 1 sample |
| # Model fits | 5 | 100 |
| Bias | Higher (less training data) | Lower (almost all data used) |
| Variance | Lower (20-sample test sets are noisy but averaged) | High (each test is 1 sample — very noisy) |
| Computation | Fast | 20× slower |

**Recommendation:** With 100 samples, **k=10** is a good middle ground. LOOCV is theoretically appealing (minimal bias) but each individual score is noisy (test on 1 sample) and they are highly correlated. The std of LOOCV scores is often misleadingly large. For n=100, k=10 gives 10 model fits and 10-sample test sets — manageable and fairly reliable.

</details>

---

**Question 3**  
You perform 5-fold CV, pick the best model (highest CV score), then report that CV score as the model's final performance. Why might this be slightly optimistic?

<details>
<summary>Click to reveal answer</summary>

This is called **selection bias** or **optimistic CV bias**. When you:
1. Try multiple models (or hyperparameter settings) on the same CV split
2. Pick the best one
3. Report its CV score as the expected performance

...you are effectively "fitting to the CV" itself. The model you selected was the one that happened to do well on this particular random partition. On truly unseen data it may not perform as well. The more models you compare, the more severe this optimism.

**Solution:** Use **nested cross-validation**:
- **Outer loop:** evaluate model selection procedure (gives unbiased generalization estimate)
- **Inner loop:** select the best model/hyperparameters

Or: hold out a completely separate **test set** that you never touch during model selection, and use that for the final performance report.

</details>